# Polyscope U-Net result viewer

Interactively compare the full 3D fields produced by `02_constraint_experiments.ipynb`.

In [ ]:
%pip install -q numpy scipy pandas scikit-image polyscope

In [13]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import polyscope as ps
import polyscope.imgui as psim
from scipy.ndimage import map_coordinates
from skimage.measure import marching_cubes

ROOT = Path(".") if Path("data").is_dir() else Path("mit-crisscross")
DATA = ROOT / "data/bunny_cross_sections.npz"
PREDICTIONS = ROOT / "data/constraint_predictions_64.npz"
METRICS = ROOT / "data/constraint_metrics.csv"
MESH_METRICS = ROOT / "data/crosssdf_paper_metrics.csv"
METHODS = ["none", "soft", "hard", "eikonal", "crosssdf"]
PLANE_RESOLUTION = 70
PLANE_PADDING = 0.12
SDF_LIMIT = 0.15
BACKEND = os.environ.get("POLYSCOPE_BACKEND", "auto")

## Load predictions, observations, and metrics

In [15]:
for path in (DATA, PREDICTIONS, METRICS, MESH_METRICS):
    assert path.is_file(), path

with np.load(PREDICTIONS) as archive:
    target = archive["target"].squeeze().astype(np.float32)
    predictions = {name: archive[name].squeeze().astype(np.float32) for name in METHODS}

with np.load(DATA) as archive:
    grid_axis = archive["grid_axis"].astype(np.float32)
    plane_normals = archive["plane_normals"].astype(np.float32)
    plane_origins = archive["plane_origins"].astype(np.float32)
    sdf2d = archive["sdf2d"].astype(np.float32)
    contour_vertices = archive["contour_vertices"].astype(np.float32)
    contour_edges = archive["contour_edges"].astype(np.int32)
    contour_plane = archive["contour_plane"].astype(np.int16)

metrics = pd.read_csv(METRICS, index_col=0)
mesh_metrics = pd.read_csv(MESH_METRICS, index_col=0)
overview = metrics.loc[METHODS, ["iou", "surface_mae", "eikonal_error"]].join(
    mesh_metrics.loc[METHODS, ["chamfer_x100", "hausdorff_x100", "components"]]
)
print({"prediction_shape": target.shape, "methods": METHODS, "observed_planes": len(plane_normals)})
overview

{'prediction_shape': (64, 64, 64), 'methods': ['none', 'soft', 'hard', 'eikonal', 'crosssdf'], 'observed_planes': 6}


,iou,surface_mae,eikonal_error,chamfer_x100,hausdorff_x100,components
none,0.992986,0.000984,0.345812,1.266270,4.693785,1.0
soft,0.990385,0.001304,0.403419,1.263556,4.335008,1.0
hard,0.995906,0.000590,0.347884,1.250027,4.322883,1.0
eikonal,0.995692,0.000690,0.014763,1.266876,4.327712,1.0
crosssdf,0.586155,0.052827,0.119510,8.891017,62.262619,1.0


## Extract zero-level surfaces

In [17]:
spacing = float(grid_axis[1] - grid_axis[0])
grid_min = float(grid_axis[0])

def extract_surface(field):
    if field.min() > 0 or field.max() < 0:
        raise ValueError("Field has no zero-level set")
    vertices, faces, _, _ = marching_cubes(field, level=0, spacing=(spacing,) * 3)
    return vertices + grid_min, faces.astype(np.int32)


def sample_field(field, points):
    coordinates = ((points - grid_min) / spacing).T
    return map_coordinates(field, coordinates, order=1, mode="nearest").astype(np.float32)


target_vertices, target_faces = extract_surface(target)
surface_data = {}
for method, field in predictions.items():
    vertices, faces = extract_surface(field)
    signed_offset = sample_field(target, vertices)
    surface_data[method] = {
        "vertices": vertices,
        "faces": faces,
        "absolute target-SDF error": np.abs(signed_offset),
        "signed target SDF": signed_offset,
    }

surface_errors = np.concatenate([surface_data[name]["absolute target-SDF error"] for name in METHODS])
SURFACE_ERROR_LIMIT = max(float(np.quantile(surface_errors, 0.98)), 1e-6)
print({name: (len(data["vertices"]), len(data["faces"])) for name, data in surface_data.items()})

{'none': (11016, 22028), 'soft': (11042, 22080), 'hard': (11030, 22056), 'eikonal': (10982, 21960), 'crosssdf': (11268, 22536)}


## Sample each prediction on the observed planes

In [19]:
def plane_frame(normal):
    reference = np.eye(3)[np.argmin(np.abs(normal))]
    u = np.cross(normal, reference)
    u /= np.linalg.norm(u)
    v = np.cross(normal, u)
    return np.column_stack((u, v))


def grid_faces(rows, columns):
    row, column = np.meshgrid(np.arange(rows - 1), np.arange(columns - 1), indexing="ij")
    lower_left = (row * columns + column).ravel()
    return np.vstack((
        np.column_stack((lower_left, lower_left + 1, lower_left + columns)),
        np.column_stack((lower_left + 1, lower_left + columns + 1, lower_left + columns)),
    )).astype(np.int32)


def plane_contours(plane_id):
    vertex_ids = np.flatnonzero(contour_plane == plane_id)
    remap = np.full(len(contour_vertices), -1, dtype=np.int32)
    remap[vertex_ids] = np.arange(len(vertex_ids))
    edge_mask = np.isin(contour_edges[:, 0], vertex_ids) & np.isin(contour_edges[:, 1], vertex_ids)
    return contour_vertices[vertex_ids], remap[contour_edges[edge_mask]]


def build_plane(plane_id):
    normal = plane_normals[plane_id]
    origin = plane_origins[plane_id]
    frame = plane_frame(normal)
    curve_vertices, _ = plane_contours(plane_id)
    curve_uv = (curve_vertices - origin) @ frame
    lower = curve_uv.min(axis=0) - PLANE_PADDING
    upper = curve_uv.max(axis=0) + PLANE_PADDING
    u = np.linspace(lower[0], upper[0], PLANE_RESOLUTION)
    v = np.linspace(lower[1], upper[1], PLANE_RESOLUTION)
    uv = np.stack(np.meshgrid(u, v, indexing="xy"), axis=-1)
    vertices = origin + uv.reshape(-1, 2) @ frame.T
    return vertices.astype(np.float32), grid_faces(PLANE_RESOLUTION, PLANE_RESOLUTION)


plane_parts = [build_plane(i) for i in range(len(plane_normals))]
plane_vertices = np.vstack([vertices for vertices, _ in plane_parts])
plane_faces = []
offset = 0
for vertices, faces in plane_parts:
    plane_faces.append(faces + offset)
    offset += len(vertices)
plane_faces = np.vstack(plane_faces)

observed_2d = np.concatenate([sample_field(sdf2d[i], plane_parts[i][0]) for i in range(len(plane_normals))])
target_on_planes = sample_field(target, plane_vertices)
plane_fields = {}
for method, field in predictions.items():
    predicted = sample_field(field, plane_vertices)
    plane_fields[method] = {
        "observed 2D SDF": observed_2d,
        "predicted 3D SDF": predicted,
        "absolute 3D SDF error": np.abs(predicted - target_on_planes),
        "2D-bound excess": np.maximum(0, np.abs(predicted) - np.abs(observed_2d)),
    }

PLANE_ERROR_LIMIT = max(float(np.quantile(np.concatenate([plane_fields[name]["absolute 3D SDF error"] for name in METHODS]), 0.98)), 1e-6)
PLANE_BOUND_LIMIT = max(float(np.quantile(np.concatenate([plane_fields[name]["2D-bound excess"] for name in METHODS]), 0.995)), 1e-6)
print({"plane_vertices": len(plane_vertices), "plane_faces": len(plane_faces)})

{'plane_vertices': 29400, 'plane_faces': 57132}


## Register the scene

In [23]:
ps.set_allow_headless_backends(True)
ps.init(BACKEND)
ps.remove_all_structures()
ps.set_window_size(1280, 900)
ps.set_up_dir("y_up")
ps.set_front_dir("neg_z_front")
ps.set_ground_plane_mode("none")
ps.set_background_color((1.0, 1.0, 1.0))
ps.set_transparency_mode("pretty")
ps.set_transparency_render_passes(12)

target_handle = ps.register_surface_mesh("target surface", target_vertices, target_faces, smooth_shade=True)
target_handle.set_color((0.42, 0.42, 0.42))
target_handle.set_transparency(0.72)
target_handle.set_edge_width(0.0)

result_handles = {}
plane_handles = {}
for method in METHODS:
    data = surface_data[method]
    result = ps.register_surface_mesh(f"prediction: {method}", data["vertices"], data["faces"], smooth_shade=True)
    result.set_color((0.20, 0.48, 0.82))
    result.set_edge_width(0.0)
    result_handles[method] = result

    planes = ps.register_surface_mesh(f"observed planes: {method}", plane_vertices, plane_faces, smooth_shade=False)
    planes.set_transparency(0.48)
    planes.set_edge_width(0.0)
    plane_handles[method] = planes

contour_handle = ps.register_curve_network("observed contours", contour_vertices, contour_edges)
contour_handle.set_color((0.03, 0.03, 0.03))
contour_handle.set_radius(0.005, relative=False)

SURFACE_MODES = ["absolute target-SDF error", "signed target SDF", "solid color"]
PLANE_MODES = ["predicted 3D SDF", "observed 2D SDF", "absolute 3D SDF error", "2D-bound excess"]
state = {"method": 0, "surface_mode": 0, "plane_mode": 0, "show_target": True, "show_contours": True, "show_planes": False}

def style_surface(method):
    handle = result_handles[method]
    handle.remove_all_quantities()
    mode = SURFACE_MODES[state["surface_mode"]]
    if mode == "absolute target-SDF error":
        handle.add_scalar_quantity(mode, surface_data[method][mode], defined_on="vertices", cmap="viridis", vminmax=(0, SURFACE_ERROR_LIMIT), enabled=True)
    elif mode == "signed target SDF":
        handle.add_scalar_quantity(mode, surface_data[method][mode], defined_on="vertices", datatype="symmetric", cmap="coolwarm", vminmax=(-SURFACE_ERROR_LIMIT, SURFACE_ERROR_LIMIT), enabled=True)


def style_planes(method):
    handle = plane_handles[method]
    handle.remove_all_quantities()
    mode = PLANE_MODES[state["plane_mode"]]
    values = plane_fields[method][mode]
    if mode in ("predicted 3D SDF", "observed 2D SDF"):
        handle.add_scalar_quantity(mode, values, defined_on="vertices", datatype="symmetric", cmap="coolwarm", vminmax=(-SDF_LIMIT, SDF_LIMIT), enabled=True)
    else:
        limit = PLANE_ERROR_LIMIT if mode == "absolute 3D SDF error" else PLANE_BOUND_LIMIT
        handle.add_scalar_quantity(mode, values, defined_on="vertices", cmap="viridis", vminmax=(0, limit), enabled=True)


def apply_state():
    active = METHODS[state["method"]]
    for method in METHODS:
        result_handles[method].set_enabled(method == active)
        plane_handles[method].set_enabled(state["show_planes"] and method == active)
    target_handle.set_enabled(state["show_target"])
    contour_handle.set_enabled(state["show_contours"])
    style_surface(active)
    style_planes(active)


apply_state()

## Open the interactive viewer

Use the `U-Net results` panel to change the loss condition, surface coloring, overlays, and observed-plane quantity. Negative signed target SDF means the predicted surface lies inside the target; positive means it lies outside.

## view meshes/cross section slices

In [ ]:
def ui_callback():
    psim.TextUnformatted("U-Net results")
    changed, value = psim.Combo("Loss condition", state["method"], METHODS)
    if changed:
        state["method"] = value
        apply_state()

    if psim.Button("< Previous"):
        state["method"] = (state["method"] - 1) % len(METHODS)
        apply_state()
    psim.SameLine()
    if psim.Button("Next >"):
        state["method"] = (state["method"] + 1) % len(METHODS)
        apply_state()

    changed, value = psim.Combo("Surface coloring", state["surface_mode"], SURFACE_MODES)
    if changed:
        state["surface_mode"] = value
        style_surface(METHODS[state["method"]])

    changed, value = psim.Checkbox("Show target", state["show_target"])
    if changed:
        state["show_target"] = value
        target_handle.set_enabled(value)
    changed, value = psim.Checkbox("Show observed contours", state["show_contours"])
    if changed:
        state["show_contours"] = value
        contour_handle.set_enabled(value)
    changed, value = psim.Checkbox("Show observed planes", state["show_planes"])
    if changed:
        state["show_planes"] = value
        plane_handles[METHODS[state["method"]]].set_enabled(value)

    changed, value = psim.Combo("Plane quantity", state["plane_mode"], PLANE_MODES)
    if changed:
        state["plane_mode"] = value
        style_planes(METHODS[state["method"]])

    method = METHODS[state["method"]]
    field_row = metrics.loc[method]
    mesh_row = mesh_metrics.loc[method]
    psim.Separator()
    psim.TextUnformatted(f"IoU: {field_row['iou']:.4f}")
    psim.TextUnformatted(f"Surface MAE: {field_row['surface_mae']:.6f}")
    psim.TextUnformatted(f"Eikonal error: {field_row['eikonal_error']:.5f}")
    psim.TextUnformatted(f"Chamfer x100: {mesh_row['chamfer_x100']:.4f}")
    psim.TextUnformatted(f"Hausdorff x100: {mesh_row['hausdorff_x100']:.4f}")
    psim.TextUnformatted(f"Components: {int(mesh_row['components'])}")

    if psim.Button("Reset camera"):
        ps.reset_camera_to_home_view()


ps.set_user_callback(ui_callback)
ps.reset_camera_to_home_view()
ps.show(1 if BACKEND == "openGL_mock" else None)

In [25]:
ps.remove_all_structures()
ps.remove_all_slice_planes()

volume_dims = tuple(int(n) for n in target.shape)
volume_low = (float(grid_axis[0]),) * 3
volume_high = (float(grid_axis[-1]),) * 3
volume_error_limit = max(float(np.quantile(np.concatenate([np.abs(field - target).ravel() for field in predictions.values()]), 0.99)), 1e-6)
volume_handles = {}

target_volume = ps.register_volume_grid("target volume", volume_dims, volume_low, volume_high, enabled=False, edge_width=0.0)
target_volume.add_scalar_quantity("SDF", target, defined_on="nodes", datatype="symmetric", cmap="coolwarm", vminmax=(-SDF_LIMIT, SDF_LIMIT), enabled=True, enable_gridcube_viz=True, enable_isosurface_viz=True, isosurface_level=0.0, isosurface_color=(0.42, 0.42, 0.42), slice_planes_affect_isosurface=False)

for index, method in enumerate(METHODS):
    field = predictions[method]
    grid = ps.register_volume_grid(f"prediction volume: {method}", volume_dims, volume_low, volume_high, enabled=index == 0, edge_width=0.0)
    grid.add_scalar_quantity("SDF", field, defined_on="nodes", datatype="symmetric", cmap="coolwarm", vminmax=(-SDF_LIMIT, SDF_LIMIT), enabled=True, enable_gridcube_viz=True, enable_isosurface_viz=True, isosurface_level=0.0, isosurface_color=(0.20, 0.48, 0.82), slice_planes_affect_isosurface=False)
    grid.add_scalar_quantity("absolute field error", np.abs(field - target), defined_on="nodes", cmap="viridis", vminmax=(0, volume_error_limit), enabled=False)
    grid.add_scalar_quantity("signed field error", field - target, defined_on="nodes", datatype="symmetric", cmap="coolwarm", vminmax=(-volume_error_limit, volume_error_limit), enabled=False)
    volume_handles[method] = grid

ps.add_scene_slice_plane()
volume_state = {"method": 0, "show_target": False}

def apply_volume_state():
    active = METHODS[volume_state["method"]]
    for method, handle in volume_handles.items():
        selected = method == active
        handle.set_enabled(selected)
        handle.set_transform_gizmo_enabled(selected)
    target_volume.set_enabled(volume_state["show_target"])
    target_volume.set_transform_gizmo_enabled(False)


def volume_ui_callback():
    psim.TextUnformatted("U-Net volume grids")
    changed, value = psim.Combo("Loss condition", volume_state["method"], METHODS)
    if changed:
        volume_state["method"] = value
        apply_volume_state()

    if psim.Button("< Previous"):
        volume_state["method"] = (volume_state["method"] - 1) % len(METHODS)
        apply_volume_state()
    psim.SameLine()
    if psim.Button("Next >"):
        volume_state["method"] = (volume_state["method"] + 1) % len(METHODS)
        apply_volume_state()

    changed, value = psim.Checkbox("Show target volume", volume_state["show_target"])
    if changed:
        volume_state["show_target"] = value
        target_volume.set_enabled(value)

    method = METHODS[volume_state["method"]]
    field_row = metrics.loc[method]
    mesh_row = mesh_metrics.loc[method]
    psim.Separator()
    psim.TextUnformatted("Select SDF or an error quantity in the grid panel.")
    psim.TextUnformatted(f"IoU: {field_row['iou']:.4f}")
    psim.TextUnformatted(f"Surface MAE: {field_row['surface_mae']:.6f}")
    psim.TextUnformatted(f"Eikonal error: {field_row['eikonal_error']:.5f}")
    psim.TextUnformatted(f"Chamfer x100: {mesh_row['chamfer_x100']:.4f}")
    psim.TextUnformatted(f"Hausdorff x100: {mesh_row['hausdorff_x100']:.4f}")

    if psim.Button("Reset camera"):
        ps.reset_camera_to_home_view()


apply_volume_state()
ps.set_user_callback(volume_ui_callback)
ps.reset_camera_to_home_view()
ps.show(1 if BACKEND == "openGL_mock" else None)